# FHIR Patient Bronze-to-Silver Transformation

## Purpose

In this notebook, I transform raw FHIR Patient resources from the Bronze layer
into a structured and analytics-ready Silver Delta table.

### Source

`health_insurance.bronze.fhir_patient_raw`

### Target

`health_insurance.silver.fhir_patient`

### What I am doing in this transformation

- I parse the raw FHIR JSON stored in Bronze.
- I infer the combined Patient schema using a Serverless-compatible approach.
- I extract nested Patient attributes from FHIR `STRUCT` and `ARRAY` fields.
- I safely handle optional or empty FHIR arrays without failing the transformation.
- I extract the official patient name, primary address, phone number, and medical record number.
- I standardize selected dates and categorical values.
- I derive patient age and age groups.
- I preserve ingestion metadata so lineage is retained from Bronze to Silver.
- I add a Silver transformation timestamp.

I am keeping formal data-quality enforcement outside this notebook because
those rules will be attached to the production Lakeflow transformation later.
This notebook focuses on structural parsing, standardization, and conformance.




In [0]:
# defining the source and target tables used by this transformation.

CATALOG = "health_insurance"

SOURCE_TABLE = f"{CATALOG}.bronze.fhir_patient_raw"
TARGET_TABLE = f"{CATALOG}.silver.fhir_patient"

print("Source:", SOURCE_TABLE)
print("Target:", TARGET_TABLE)


In [0]:
# loading the Bronze FHIR Patient table and inspecting its current shape.

patient_bronze_df = spark.table(SOURCE_TABLE)

print(f"Rows: {patient_bronze_df.count():,}")
print(f"Columns: {len(patient_bronze_df.columns)}")

patient_bronze_df.printSchema()

display(patient_bronze_df.limit(5))


## Inferring the FHIR Patient JSON schema

Because I am using Databricks Serverless, I avoid Spark RDD APIs.

I infer the schema directly from the Bronze `raw_json` column with
`schema_of_json_agg`. I use the aggregate form because FHIR fields are optional
and can vary between Patient resources. This lets Databricks build a combined
schema from the Patient records currently present in Bronze.


In [0]:
# inferring a combined FHIR Patient schema without using RDD operations.

schema_result = spark.sql(f'''
    SELECT schema_of_json_agg(raw_json) AS patient_schema
    FROM {SOURCE_TABLE}
''').first()

patient_schema = schema_result["patient_schema"]

print("FHIR Patient schema:")
print(patient_schema)


In [0]:
# parsing each raw JSON string into a structured Spark column.

from pyspark.sql import functions as F

patient_parsed_df = (
    patient_bronze_df
    .withColumn(
        "patient",
        F.from_json(
            F.col("raw_json"),
            patient_schema
        )
    )
)

patient_parsed_df.select("patient.*").printSchema()


In [0]:
#  inspecting several nested Patient attributes before flattening them.

display(
    patient_parsed_df.select(
        F.col("patient.id").alias("patient_id"),
        F.col("patient.gender").alias("gender"),
        F.col("patient.birthDate").alias("birth_date"),
        F.col("patient.name").alias("names"),
        F.col("patient.address").alias("addresses"),
        F.col("patient.telecom").alias("telecom")
    ).limit(10)
)


## Extracting the core Patient attributes

At this point I keep the nested arrays that I still need for later extraction.
I only select fields that exist in the FHIR Patient schema currently present in
Bronze.

FHIR fields are optional, so I do not assume that every Patient contains every
possible attribute.


In [0]:
#  selecting the core Patient attributes and the nested arrays I still need.

patient_core_df = (
    patient_parsed_df
    .select(
        F.col("patient.id").alias("patient_id"),
        F.col("patient.gender").alias("gender"),
        F.to_date(F.col("patient.birthDate")).alias("birth_date"),

        F.col("patient.name").alias("name"),
        F.col("patient.address").alias("address"),
        F.col("patient.telecom").alias("telecom"),
        F.col("patient.identifier").alias("identifier"),

        F.col("patient.communication").alias("communication"),
        F.col("patient.maritalStatus").alias("marital_status"),
        F.to_timestamp(F.col("patient.deceasedDateTime")).alias("deceased_datetime"),

        "_ingested_at",
        "_source_system",
        "_resource_type"
    )
)


## Extracting the official Patient name safely

FHIR allows the `name` field to contain multiple values, such as official,
maiden, or other names.

I first filter the array to `use = 'official'`. I then use `get(array, 0)`
instead of positional `element_at()` access. `get()` returns `NULL` when an
array is empty, which prevents an optional FHIR array from failing the entire
Silver transformation.


In [0]:
# extracting the official Patient name with null-safe array access.

patient_name_df = (
    patient_core_df

    .withColumn(
        "official_names",
        F.expr("""
            filter(
                name,
                x -> x.use = 'official'
            )
        """)
    )

    .withColumn(
        "official_name",
        F.expr("get(official_names, 0)")
    )

    .withColumn(
        "given_name",
        F.expr("get(official_name.given, 0)")
    )

    .withColumn(
        "family_name",
        F.col("official_name.family")
    )
)


## Extracting the primary Patient address safely

FHIR also allows multiple addresses. For this Silver patient entity, I keep one
row per Patient and use the first available address as the primary analytical
address.

I again use null-safe `get()` access so a Patient with no address produces
`NULL` values rather than causing the transformation to fail.


In [0]:
# extracting the first available address and its main attributes safely.

patient_address_df = (
    patient_name_df

    .withColumn(
        "primary_address",
        F.expr("get(address, 0)")
    )

    .withColumn(
        "address_line",
        F.expr("get(primary_address.line, 0)")
    )

    .withColumn(
        "city",
        F.col("primary_address.city")
    )

    .withColumn(
        "state",
        F.col("primary_address.state")
    )

    .withColumn(
        "postal_code",
        F.col("primary_address.postalCode")
    )

    .withColumn(
        "country",
        F.col("primary_address.country")
    )
)


## Extracting the Patient phone number

FHIR `telecom` is an array and can contain several contact types. I do not
assume the first element is a phone number.

I filter the array to entries where `system = 'phone'` and then safely select
the first matching value.


In [0]:
# extracting the first available phone contact safely.

patient_contact_df = (
    patient_address_df

    .withColumn(
        "phone_contacts",
        F.expr("""
            filter(
                telecom,
                x -> x.system = 'phone'
            )
        """)
    )

    .withColumn(
        "phone",
        F.expr("get(phone_contacts.value, 0)")
    )
)


## Selecting an appropriate Patient identifier

The raw synthetic FHIR Patient resources can contain identifiers such as a
medical record number, Social Security number, driver's licence, and passport
number.

For my normal Silver analytical Patient table, I deliberately keep the medical
record number and do not propagate the more sensitive SSN, passport, or
driver's-licence values.

This also gives me a clear governance boundary that I can later demonstrate
with Unity Catalog security controls.


In [0]:
# extracting only the Medical Record Number from the FHIR identifier array.

patient_identifier_df = (
    patient_contact_df

    .withColumn(
        "medical_record_identifiers",
        F.expr("""
            filter(
                identifier,
                x -> exists(
                    x.type.coding,
                    c -> c.code = 'MR'
                )
            )
        """)
    )

    .withColumn(
        "medical_record_number",
        F.expr("get(medical_record_identifiers.value, 0)")
    )
)


## Deriving Patient age and age group

I calculate age from `birth_date` so downstream analytics do not need to repeat
the calculation.

I also create a standardized age-group attribute. If `birth_date` is missing,
I keep the derived age group as `UNKNOWN` instead of incorrectly assigning the
Patient to an age band.


In [0]:
# deriving the Patient age from birth_date.

patient_enriched_df = (
    patient_identifier_df
    .withColumn(
        "age",
        F.floor(
            F.months_between(
                F.current_date(),
                F.col("birth_date")
            ) / 12
        ).cast("int")
    )
)


In [0]:
# assigning a standardized age group while preserving missing ages explicitly.

patient_enriched_df = (
    patient_enriched_df
    .withColumn(
        "age_group",
        F.when(F.col("age").isNull(), "UNKNOWN")
         .when(F.col("age") < 18, "UNDER_18")
         .when(F.col("age") < 35, "18_34")
         .when(F.col("age") < 50, "35_49")
         .when(F.col("age") < 65, "50_64")
         .otherwise("65_PLUS")
    )
)


## Standardizing Patient text attributes

I normalize selected text fields so the Silver table has consistent values for
downstream joins and analytics.

I am standardizing representation only. I am not rejecting records here;
formal quality rules will be enforced when the transformation is wrapped in the
Lakeflow pipeline.


In [0]:
# standardizing selected Patient text attributes.

patient_standardized_df = (
    patient_enriched_df

    .withColumn(
        "gender",
        F.upper(F.trim("gender"))
    )

    .withColumn(
        "city",
        F.initcap(F.trim("city"))
    )

    .withColumn(
        "state",
        F.upper(F.trim("state"))
    )

    .withColumn(
        "country",
        F.upper(F.trim("country"))
    )
)


## Building the final Silver Patient dataset

I now remove the temporary nested arrays and helper columns used during parsing
and keep the conformed fields needed by the Silver patient entity.

I preserve the Bronze ingestion metadata and add
`_silver_transformed_at` so I can trace when each row reached the Silver layer.


In [0]:
# building the final conformed Silver Patient dataset.

patient_silver_df = (
    patient_standardized_df

    .select(
        "patient_id",
        "medical_record_number",

        "given_name",
        "family_name",

        "gender",
        "birth_date",
        "age",
        "age_group",

        "phone",

        "address_line",
        "city",
        "state",
        "postal_code",
        "country",

        "_source_system",
        "_resource_type",
        "_ingested_at"
    )

    .withColumn(
        "_silver_transformed_at",
        F.current_timestamp()
    )
)


In [0]:
# inspecting the final Silver Patient schema and a small result sample.

patient_silver_df.printSchema()

display(
    patient_silver_df.limit(20)
)


## Profiling optional Patient attributes

Before persisting the table, I profile several optional attributes so I can see
how much information is missing without rejecting or modifying any records.

These are descriptive checks only. I will convert the appropriate business and
technical requirements into Lakeflow expectations during the data-quality and
pipeline stage.


In [0]:
# profiling missing optional attributes without filtering any records.

patient_silver_df.select(
    F.sum(F.col("given_name").isNull().cast("int")).alias("missing_given_name"),
    F.sum(F.col("family_name").isNull().cast("int")).alias("missing_family_name"),
    F.sum(F.col("phone").isNull().cast("int")).alias("missing_phone"),
    F.sum(F.col("city").isNull().cast("int")).alias("missing_city"),
    F.sum(
        F.col("medical_record_number").isNull().cast("int")
    ).alias("missing_medical_record_number")
).show()


In [0]:
# reconciling Bronze and Silver row counts before persistence.

bronze_count = patient_bronze_df.count()
silver_count = patient_silver_df.count()

print(f"Bronze Patients: {bronze_count:,}")
print(f"Silver Patients: {silver_count:,}")
print(f"Difference: {bronze_count - silver_count:,}")


## Persisting the Silver table

At this development stage, I overwrite the Silver table so the notebook remains
repeatable while I am validating the transformation logic.

When I productionize this logic in Lakeflow, the execution pattern will become
incremental and quality expectations will be attached to the transformation.
The field-parsing and conformance logic in this notebook is intended to remain
the same.


In [0]:
# persisting the conformed Patient dataset as a Delta table in Unity Catalog.

(
    patient_silver_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", True)
    .saveAsTable(TARGET_TABLE)
)

print(f"Created Silver table: {TARGET_TABLE}")


In [0]:
%sql
-- verifying the number of Patient records written to the Silver table.

SELECT COUNT(*) AS patient_count
FROM health_insurance.silver.fhir_patient;


In [0]:
%sql
-- reviewing the distribution of patients by gender and age group as a sanity check.

SELECT
    gender,
    age_group,
    COUNT(*) AS patient_count
FROM health_insurance.silver.fhir_patient
GROUP BY gender, age_group
ORDER BY gender, age_group;


## Transformation Result

successfully transformed the raw FHIR Patient resources from Bronze into a
structured Silver Delta table.

### Source

`health_insurance.bronze.fhir_patient_raw`

### Target

`health_insurance.silver.fhir_patient`

### Transformations I applied

- I inferred the combined FHIR Patient schema using a Serverless-compatible
  DataFrame/SQL approach.
- I parsed `raw_json` into structured Spark data.
- I extracted Patient identifiers, names, demographics, contact information,
  and address attributes.
- I used null-safe array access so optional or empty FHIR arrays do not fail
  the transformation.
- I deliberately excluded highly sensitive SSN, passport, and driver's-licence
  identifiers from the normal Silver analytical model.
- I retained the medical record number as the operational Patient identifier.
- I converted the Patient birth date to a proper date type.
- I derived age and standardized age groups.
- I normalized selected categorical and text attributes.
- I preserved Bronze lineage metadata and added a Silver transformation
  timestamp.
- I reconciled Bronze and Silver row counts before persistence.

### Data-quality boundary

I am not rejecting records in this notebook.

Missing optional attributes are allowed to become `NULL`, while formal rules
such as required Patient IDs, acceptable age ranges, and other quality
conditions will be attached as Lakeflow expectations when I productionize the
Bronze-to-Silver pipeline.

### Architecture

SMART FHIR API  
↓  
Auto Loader-managed Bronze ingestion  
↓  
`health_insurance.bronze.fhir_patient_raw`  
↓  
FHIR parsing and conformance  
↓  
`health_insurance.silver.fhir_patient`  
↓  
Lakeflow expectations / validated Silver  
↓  
Gold analytical model



### Status

**FHIR Patient Bronze-to-Silver transformation: COMPLETE**
